# MATE Move-Selection: Local Gemma 4 E2B, Thinking ON (100 positions)

Gemma 4 E2B (4-bit, T4, **no API**) on the **exact same positions** the
DeepSeek thinking-final arm scored (`results/mate-selection-thinking100-final/`,
94/100). Same forced answer prompt, same 32768 total budget. Thinking runs
locally via the `<|channel>thought` channel; the thinking text and reasoning
tokens are recorded, and samples/summary/report upload to Hugging Face
(vedangfake/chess-bench-results) exactly like the DeepSeek runs.

**Flow: probe 5 positions first, inspect the parsing/extraction, then run the
full 100.** `--resume` means the full run skips the probed 5. Streams to
the gemma dashboard page (chess-bench-live.pages.dev/gemma.html).

Secrets needed: `GITHUB_TOKEN` (clone + live push), `HF_WRITE_TOKEN` (results
archive). `google/gemma-4-E2B-it` is NOT gated, so no HF token is needed to
load the model. Attach, SAVE, then Kernel -> Restart & Run All.

## 1. Get the repo (fresh clone)

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "chess-slm-benchmark"
if REPO.exists():
    shutil.rmtree(REPO)

def find_token():
    for name in ("GITHUB_TOKEN", "GH_TOKEN"):
        if os.environ.get(name):
            return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None

token = find_token()
url = "https://github.com/Vedang-P/chess-slm-benchmark.git"
if token:
    url = url.replace("https://", f"https://x-access-token:{token}@")
# This arm lives on the mate-e2b-kaggle branch (--local-thinking and
# --live-namespace are NOT on main yet); clone the branch explicitly so the
# kernel always runs the intended code, independent of main's state.
res = subprocess.run(["git", "clone", "--quiet", "--branch", "mate-e2b-kaggle",
                      url, str(REPO)],
                     capture_output=True, text=True)
if res.returncode != 0:
    raise RuntimeError("clone failed (token not attached?): " + res.stderr[-300:])
os.chdir(REPO)
print("cwd:", Path.cwd())

## 2. Dependencies (transformers >= 5.13 for Gemma 4)

In [ ]:
import subprocess, sys
# Kaggle's free tier hands out a P100 (sm_60) OR a T4 (sm_75). Recent torch
# wheels dropped sm_60, so pin the last CUDA-12.1 build with both archs
# BEFORE anything else installs torch; bitsandbytes is pinned to the matching
# multi-CUDA build, and the requirements install afterwards must NOT clobber
# these pins (no -U: it still upgrades transformers to >=5.13 on its own).
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "torch==2.4.1", "--index-url",
                "https://download.pytorch.org/whl/cu121"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "bitsandbytes==0.44.1"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "-r", "requirements.txt"], check=True)
import torch, transformers
if int(transformers.__version__.split(".")[0]) < 5:
    raise RuntimeError(f"transformers {transformers.__version__} too old "
                       "for Gemma 4 (needs >= 5.13)")
print("transformers", transformers.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0),
          "| cap", torch.cuda.get_device_capability(0),
          "| vram GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 3. Engine/dataset gate

In [ ]:
status = subprocess.run([sys.executable, "scripts/test_engine.py", "--quick"],
                        capture_output=True, text=True)
if status.returncode != 0:
    print(status.stdout[-2000:]); print(status.stderr[-2000:])
    raise RuntimeError("test_engine failed")
print("ALL TESTS PASSED")

## 4. Probe: 5 positions through the REAL pipeline

Runs `run_mate_eval.py --local-thinking` on the first 5 ids.
This is the parsing/extraction test: does the thought channel get split into
`reasoning` + `content`? Are `reasoning_tokens` recorded? Does the answer
parser land on MoveA/MoveB? Model load is ~1-3 min; 5 positions a few
minutes. The probe writes into the SAME output dir as the full run under a
shared run id, so the archive ends up with one complete cell.

In [ ]:
import json, os, time
from pathlib import Path

# one run id for probe + full run: the archive gets ONE cell, the final
# upload overwrites the 5-position partial (upload_cell re-uploads the
# whole current samples file)
os.environ["BENCH_RUN_ID"] = "mate-selection-e2b-100-" + time.strftime("%Y%m%d")

cmd = [sys.executable, "scripts/run_mate_eval.py",
       "--model", "gemma4-e2b",
       "--ids", "mate-sel-00279,mate-sel-00477,mate-sel-00542,mate-sel-00563,mate-sel-00666",
       "--force-answer-prompt",
       "--max_new_tokens", "32768",
       "--local-thinking",
       "--output_dir", "results/mate-selection-e2b-100",
       "--live-push",
       "--live-namespace", "gemma",
       "--resume",
       "--verbose"]
t0 = time.time()
res = subprocess.run(cmd)
print(f"probe exit rc={res.returncode} after {(time.time()-t0)/60:.1f}min")
if res.returncode != 0:
    raise RuntimeError(f"probe run failed with rc={res.returncode} — fix "
                       "before proceeding")

## 5. Inspect the probe before proceeding

Verify, for each of the 5 samples: `status` is correct/wrong (NOT
no_answer/parse_error), `reasoning_chars > 0` (thinking was captured),
`token_usage.reasoning_tokens > 0` (thinking is accounted like the gateway
arm). If 0/5 parse, this cell raises and the 100-run must NOT be started.

In [ ]:
import json, collections, glob
from pathlib import Path

# the samples file is named after the MODEL (gemma4-e2b_*), not deepseek —
# glob so the inspect cell can never point at the wrong run's file
files = glob.glob("results/mate-selection-e2b-100/*_mate-selection-test_strategy.samples.jsonl")
if not files:
    raise RuntimeError(f"no samples file under results/mate-selection-e2b-100/ — probe did not write results")
rows = [json.loads(l) for l in open(files[0]) if l.strip()]
print(f"probe samples on disk: {len(rows)}")
for s in rows:
    tu = s.get("token_usage") or {}
    print(f"{s['position_id']}: {s['status']:11s} label={s.get('label')} "
          f"move={s.get('move')} correct={s.get('compliance')} "
          f"answer_chars={s.get('answer_chars')} reasoning_chars={s.get('reasoning_chars')} "
          f"tokens={{in:{tu.get('input_tokens')} out:{tu.get('output_tokens')} "
          f"reason:{tu.get('reasoning_tokens')}}}")
print()
s = rows[0]
print("--- sample 0 reasoning (first 400 chars) ---")
print((s.get("reasoning") or "")[:400])
print("--- sample 0 output ---")
print(repr((s.get("output") or "")[:200]))
parsed = [r for r in rows if r["status"] in ("correct", "wrong")]
if len(parsed) == 0:
    raise RuntimeError(
        "0/5 positions produced a parseable MoveA/MoveB choice. Parsing or "
        "extraction is broken — fix before the run. Check the "
        "reasoning/output dump above; do NOT proceed.")
print(f"\nPROBE OK: {len(parsed)}/5 parsed — extraction looks sane.")

## 6. The full 100-position run

Same command, all 100 ids, `--resume` skips the probed 5. Thinking E2B on a
T4 is slow (each position can take minutes); expect roughly 3-8h for 100.
`--live-push` streams progress to the gemma dashboard page; the HF archive
gets the complete cell when the run finishes. If the session dies, re-run
this notebook — the probe/run cells skip what is already scored.

In [ ]:
import json, os, time

cmd = [sys.executable, "scripts/run_mate_eval.py",
       "--model", "gemma4-e2b",
       "--ids", "mate-sel-00279,mate-sel-00477,mate-sel-00542,mate-sel-00563,mate-sel-00666,mate-sel-00791,mate-sel-00847,mate-sel-00874,mate-sel-01227,mate-sel-01291,mate-sel-01357,mate-sel-01441,mate-sel-01554,mate-sel-01597,mate-sel-01624,mate-sel-01655,mate-sel-01665,mate-sel-01873,mate-sel-01878,mate-sel-01959,mate-sel-02042,mate-sel-02071,mate-sel-02130,mate-sel-02229,mate-sel-02385,mate-sel-02455,mate-sel-02481,mate-sel-02612,mate-sel-02627,mate-sel-02849,mate-sel-02968,mate-sel-03005,mate-sel-03338,mate-sel-03495,mate-sel-03633,mate-sel-03719,mate-sel-03756,mate-sel-03823,mate-sel-03854,mate-sel-03936,mate-sel-03949,mate-sel-04292,mate-sel-04366,mate-sel-04745,mate-sel-05679,mate-sel-05744,mate-sel-05787,mate-sel-05943,mate-sel-06150,mate-sel-06253,mate-sel-06279,mate-sel-06464,mate-sel-06667,mate-sel-06868,mate-sel-07151,mate-sel-07154,mate-sel-07414,mate-sel-07432,mate-sel-07472,mate-sel-07482,mate-sel-07520,mate-sel-07612,mate-sel-07721,mate-sel-08022,mate-sel-08060,mate-sel-08110,mate-sel-08284,mate-sel-08356,mate-sel-08498,mate-sel-08651,mate-sel-08982,mate-sel-09036,mate-sel-09100,mate-sel-09109,mate-sel-09217,mate-sel-09342,mate-sel-09395,mate-sel-09401,mate-sel-09461,mate-sel-09666,mate-sel-09683,mate-sel-09863,mate-sel-10201,mate-sel-10203,mate-sel-09848,mate-sel-10294,mate-sel-00640,mate-sel-03955,mate-sel-04124,mate-sel-04600,mate-sel-04709,mate-sel-05255,mate-sel-06659,mate-sel-09605,mate-sel-10387,mate-sel-00543,mate-sel-01167,mate-sel-02586,mate-sel-04111,mate-sel-02999",
       "--force-answer-prompt",
       "--max_new_tokens", "32768",
       "--local-thinking",
       "--output_dir", "results/mate-selection-e2b-100",
       "--live-push",
       "--live-namespace", "gemma",
       "--resume",
       ]
t0 = time.time()
res = subprocess.run(cmd)
print(f"full run exit rc={{res.returncode}} after {{(time.time()-t0)/3600:.2f}}h")

## 7. Results vs DeepSeek

In [ ]:
import json, glob
from pathlib import Path

files = glob.glob("results/mate-selection-e2b-100/*_mate-selection-test_strategy.samples.jsonl")
if not files:
    raise RuntimeError(f"no samples file under results/mate-selection-e2b-100/")
rows = [json.loads(l) for l in open(files[0]) if l.strip()]
scored = [r for r in rows if r["status"] != "api_error"]
parsed = [r for r in scored if r["status"] in ("correct", "wrong")]
n = len(scored)
acc = {
    "n": n,
    "n_attempted": len(rows),
    "api_error": sum(r["status"] == "api_error" for r in rows),
    "parse_rate": round(len(parsed) / n, 4) if n else None,
    "accuracy_strict": round(sum(bool(r["compliance"]) for r in scored) / n, 4) if n else None,
    "accuracy_of_parsed": round(sum(bool(r["compliance"]) for r in parsed) / len(parsed), 4) if parsed else None,
    "correct": sum(bool(r["compliance"]) for r in scored),
    "wrong": sum(r["status"] == "wrong" for r in scored),
    "no_answer": sum(r["status"] == "no_answer" for r in scored),
    "parse_error": sum(r["status"] == "parse_error" for r in scored),
}
print("GEMMA 4 E2B (local, thinking ON, forced prompt, 32768 budget):")
print(json.dumps(acc, indent=1))

# DeepSeek reference numbers, saved locally 2026-08-04 (NOT fetched — the
# repo does not track results/):
DEEPSEEK_THINKING_FINAL = {"n": 100, "accuracy_strict": 0.94, "parse_rate": 1.0}
DEEPSEEK_DIRECT_1000 = {"n": 1000, "accuracy_strict": 0.486}
print()
print(f"deepseek-v4-flash thinking-final (same 100 positions): "
      f"{DEEPSEEK_THINKING_FINAL['accuracy_strict']:.0%} strict "
      f"(n={DEEPSEEK_THINKING_FINAL['n']})")
print(f"deepseek-v4-flash direct  (1000 positions):              "
      f"{DEEPSEEK_DIRECT_1000['accuracy_strict']:.0%} strict "
      f"(n={DEEPSEEK_DIRECT_1000['n']})")

## Notes
- **Confound vs DeepSeek thinking-final:** that arm had a 16384-token
  *thinking* budget within its 32768 total; local gemma has one undivided
  budget (the thinking block and answer share `max_new_tokens`). The prompt,
  position set, and total budget are identical.
- **Honesty contract:** no retries, no fallbacks. The answer is the model's
  own text after channel parsing; a budget-cut generation records its
  partial thinking as reasoning and an empty answer (no_answer, reason:
  truncated).
- **Extraction:** `processor.parse_response` (the checkpoint's shipped
  response_template) with a channel-marker fallback in src/models.py.
- Results live in `results/mate-selection-e2b-100/` and the HF archive; the repo itself
  does not track results/.